# Talking to other people's computers

**Module 3, Part 2 - APIs**

By the end of this notebook you will be able to ask a program on another
continent a question, and get an answer back into a pandas DataFrame.

We are going to take **one** API apart completely before we look at a
second one. Everything else in this session is a variation on what
happens in the first twenty minutes.

| Section | The idea |
|---|---|
| 1 | A request is just a URL |
| 2 | A response has a status *and* a body, and they are different things |
| 3 | Things go wrong, and you can see exactly how |
| 4 | JSON becomes a Python dictionary becomes a DataFrame |
| 5 | A successful response can still be wrong |
| 6 | Parameters, and why you get cut off |
| 7 | Wrappers are convenience, not magic |
| 8 | When the service needs to know who you are |
| 9 | An API that answers in prose |

## 0. Setup

Nothing here needs an account. Sections 8 and 9 do, and they say so when
we get there.

In [2]:
import json
import time
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests

print("requests", requests.__version__)
print("pandas  ", pd.__version__)

requests 2.34.2
pandas   2.3.3


### The safety net

Every call below goes through `fetch()`. It tries the real API first. If
the network is down, or the service is having a bad day, it falls back to
a copy saved on disk and **tells you loudly** that it has done so.

This is not just a teaching convenience. Falling back to cached data
silently is one of the easier ways to publish a wrong number, so the
banner matters as much as the fallback.

In [3]:
CACHE = Path("data") / "api-cache"


def fetch(url, cache_name, params=None, timeout=15):
    """Get JSON from a URL. Fall back to a saved copy if that fails."""
    try:
        response = requests.get(url, params=params, timeout=timeout)
        response.raise_for_status()
        data = response.json()
        print("LIVE:", response.url[:95])
        return data
    except Exception as problem:
        snapshot = CACHE / cache_name
        if not snapshot.exists():
            raise
        print("=" * 66)
        print("!! LIVE CALL FAILED:", type(problem).__name__)
        print("!! Using the saved copy at", snapshot)
        print("!! The numbers below are NOT current.")
        print("=" * 66)
        return json.loads(snapshot.read_text())


print("cache folder exists:", CACHE.exists())

cache folder exists: True


---

## 1. A request is just a URL

The International Space Station broadcasts its position. Someone put a
free, no-account API in front of that feed. We are going to ask it where
the station is, right now.

Here is the whole thing:

> **Predict first.** `requests.get(...)` hands something back. What do you think that object is - the text of the answer, a number, or something else?
>
> Put your answer in the chat before we run it.

In [3]:
ISS_NOW = "http://api.open-notify.org/iss-now.json"

response = requests.get(ISS_NOW, timeout=15)
response

<Response [200]>

Not the data. A **Response object** - a parcel with the answer inside it,
plus a lot of information *about* the answer. We open it in section 2.

### Reading the URL

A URL is not one thing, it is several, and every API you meet uses the
same parts. Let us make Python show us:

In [4]:
parts = urlparse(ISS_NOW)

print("scheme (how to talk)   :", parts.scheme)
print("host   (which computer):", parts.netloc)
print("path   (what to ask for):", parts.path)

scheme (how to talk)   : http
host   (which computer): api.open-notify.org
path   (what to ask for): /iss-now.json


Three of the deck's terms, in one line of output:

- **Protocol** - `http`. The set of rules both ends agreed to speak.
- **Host** - `api.open-notify.org`. A machine somewhere with an answer.
- **Endpoint** - the full address, `http://api.open-notify.org/iss-now.json`.
  One endpoint, one kind of question.

A website is a building for humans. An API is a service window on the
side of it for programs: narrower, better labelled, and it will not make
small talk.

> **Notice the `http`, not `https`.** This service does not offer an
> encrypted connection at all - try the `https://` version and it simply
> refuses. Everything we send and receive here travels in the clear.
> Fine for the position of a spacecraft that broadcasts it anyway;
> **not** fine for anything you would not put on a postcard. Real APIs
> that handle real data are `https` without exception, and section 8 uses
> one.

---

## 2. Status and body are different things

Two questions to ask of any response, in this order:

1. **Did it work?** That is the status code.
2. **What did it say?** That is the body.

Beginners collapse these into one. They are separate, and section 5 shows
why the distinction earns its keep.

In [11]:
print("status code :", response.status_code)
print("content type:", response.headers)
#print("content type:", response.headers["Content-Type"])
print("how long    :", response.elapsed.total_seconds(), "seconds")
print("status code :", response.status_code)

status code : 200
content type: {'Server': 'nginx/1.10.3', 'Date': 'Tue, 15 Sep 2026 18:23:55 GMT', 'Content-Type': 'application/json', 'Content-Length': '112', 'Connection': 'keep-alive', 'access-control-allow-origin': '*'}
how long    : 0.729693 seconds
status code : 200


`200` is the "yes, here you go" of the web. The families:

| Range | Meaning | The one you will meet |
|---|---|---|
| `1xx` | information | rare |
| `2xx` | **it worked** | `200 OK` |
| `3xx` | it moved | `301 Moved Permanently` |
| `4xx` | **you** made a mistake | `404 Not Found`, `403 Forbidden`, `429 Too Many Requests` |
| `5xx` | **they** made a mistake | `500 Internal Server Error` |

The 4xx / 5xx split is the useful one. `4xx` means fix your request.
`5xx` means wait and try again, it was not you.

### Now the body

> **Predict first.** `.content`, `.text` and `.json()` all give you the body. What Python *type* does each one hand back?
>
> Put your answer in the chat before we run it.

In [6]:
print(".content ->", type(response.content))
print(".text    ->", type(response.text))
print(".json()  ->", type(response.json()))

.content -> <class 'bytes'>
.text    -> <class 'str'>
.json()  -> <class 'dict'>


Three different things, and the difference matters:

- **`.content`** is `bytes` - the raw stream exactly as it came off the
  wire. What you want for an image or a PDF.
- **`.text`** is `str` - those bytes decoded into characters. Readable,
  but still just a string.
- **`.json()`** is a **parsed Python object**. Here a `dict`. This is the
  one you almost always want.

`.json()` is a convenience that runs `json.loads(response.text)` for you.
It is not automatic and it is not guaranteed: it only works if the body
really is JSON, and in section 3 we watch it fail.

In [7]:
response.text

'{"timestamp": 1789496635, "iss_position": {"latitude": "44.2332", "longitude": "-9.4999"}, "message": "success"}'

In [8]:
response.json()

{'timestamp': 1789496635,
 'iss_position': {'latitude': '44.2332', 'longitude': '-9.4999'},
 'message': 'success'}

In [9]:
# .text is a string, so this is string slicing, not data access:
print(repr(response.text[:60]))

# .json() is a dictionary, so this is data access:
print(response.json()["iss_position"])

'{"timestamp": 1789496635, "iss_position": {"latitude": "44.2'
{'latitude': '44.2332', 'longitude': '-9.4999'}


In [16]:
body = response.json()

assert isinstance(response.content, bytes)
assert isinstance(response.text, str)
assert isinstance(body, dict)
assert set(body) == {"iss_position", "message", "timestamp"}, body.keys()

print("Checks passed. Keys we got:", sorted(body))

Checks passed. Keys we got: ['iss_position', 'message', 'timestamp']


---

## 3. Watching it go wrong

Ask the same host for something that is not there. The host is real, so
we will get a reply - just not a happy one.

> **Predict first.** We ask for a path that does not exist. Does `requests` raise an error, or return a response?
>
> Put your answer in the chat before we run it.

In [10]:
missing = requests.get("http://api.open-notify.org/not-a-real-endpoint",timeout=15)

print("status code:", missing.status_code)
print("body length:", len(missing.text), "characters")
print("body       :", repr(missing.text[:80]))

status code: 404
body length: 0 characters
body       : ''


**No exception.** A 404 is a perfectly successful conversation in which
the answer happened to be "no". `requests` only raises when it cannot
have the conversation at all - no network, bad hostname, timeout.

This is the single most common API bug: assuming that because the code
ran, the data arrived.

Now watch `.json()` meet a body that is not JSON.

In [12]:
missing.json()

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

`JSONDecodeError`. There was nothing to parse. If you had written

```python
data = requests.get(url).json()["iss_position"]
```

this is where your program stops, and the traceback points at the
parsing, not at the real problem - which is that you asked for the
wrong thing.

### Check the status first, every time

In [13]:
def get_json(url, params=None, timeout=15):
    """Fetch JSON, but check the status before trusting the body.

    Returns (worked, data). `data` is None when it did not work.
    """
    response = requests.get(url, params=params, timeout=timeout)

    if response.status_code != 200:
        print("Request failed with", response.status_code,
              "-", response.reason)
        return False, None

    return True, response.json()


worked, data = get_json(ISS_NOW)
print("worked:", worked, "| message:", data["message"])

worked, data = get_json("http://api.open-notify.org/not-a-real-endpoint")
print("worked:", worked, "| data:", data)

worked: True | message: success
Request failed with 404 - Not Found
worked: False | data: None


In [ ]:
worked, _ = get_json("http://api.open-notify.org/not-a-real-endpoint")
assert worked is False, "a 404 must not be reported as success"
print("Check passed: the handler refuses to treat 404 as success.")

---

## 4. JSON to dictionary to DataFrame

JSON is a text format for nested data. `requests` turns it into Python
dictionaries and lists, and from there pandas is one step away.

In [14]:
now = fetch(ISS_NOW, "iss-now.json")
now

!! LIVE CALL FAILED: ConnectTimeout
!! Using the saved copy at data/api-cache/iss-now.json
!! The numbers below are NOT current.


{'message': 'success',
 'iss_position': {'latitude': '-42.3669', 'longitude': '-110.7961'},
 'timestamp': 1789327720}

In [15]:
position = now["iss_position"]

latitude = float(position["latitude"])
longitude = float(position["longitude"])

print("latitude :", latitude)
print("longitude:", longitude)

latitude : -42.3669
longitude: -110.7961


Note the `float(...)`. The API sent those numbers **as strings** - look at
the quotes in the output above. That is common, it is allowed, and if you
skip the conversion your arithmetic silently turns into string
concatenation. Always look at the types you actually received.

In [16]:
print("as received:", type(position["latitude"]), repr(position["latitude"]))
print("after float:", type(latitude), latitude)

assert isinstance(position["latitude"], str)
assert isinstance(latitude, float)
print("Check passed.")

as received: <class 'str'> '-42.3669'
after float: <class 'float'> -42.3669
Check passed.


### The timestamp

`timestamp` is a Unix time: whole seconds since 1 January 1970, UTC.
Machines like it because it is one integer with no timezone argument
attached.

In [17]:
stamp = pd.to_datetime(now["timestamp"], unit="s", utc=True)

print("raw        :", now["timestamp"])
print("as UTC     :", stamp)
print("in NZ time :", stamp.tz_convert("Pacific/Auckland"))

raw        : 1789327720
as UTC     : 2026-09-13 19:28:40+00:00
in NZ time : 2026-09-14 07:28:40+12:00


### Take several readings

One reading is a fact. Several readings are data. We will sample the
position a few times and watch the station move.

In [ ]:
SAMPLES = 6
GAP_SECONDS = 10       # longer gaps: see the note under the speed estimate

readings = []

for sample_number in range(SAMPLES):
    snapshot = fetch(ISS_NOW, "iss-now.json")
    readings.append({
        "when": pd.to_datetime(snapshot["timestamp"], unit="s", utc=True),
        "latitude": float(snapshot["iss_position"]["latitude"]),
        "longitude": float(snapshot["iss_position"]["longitude"]),
    })
    if sample_number < SAMPLES - 1:
        time.sleep(GAP_SECONDS)

track = pd.DataFrame(readings)
track

That is the whole journey: **a URL, a response, a dictionary, a
DataFrame.** Everything else in this notebook is that same path with
extra steps bolted on.

### How fast is it going?

We have positions and times, so we can work it out rather than look it
up. Distance between two points on a sphere uses the haversine formula:

$$
d = 2R\arcsin\sqrt{
  \sin^{2}\!\left(\frac{\varphi_2-\varphi_1}{2}\right)
  + \cos\varphi_1\cos\varphi_2
    \sin^{2}\!\left(\frac{\lambda_2-\lambda_1}{2}\right)}
$$

where $\varphi$ is latitude, $\lambda$ is longitude, both in radians,
and $R$ is the radius of the sphere.

In [ ]:
import math


def haversine_km(lat1, lon1, lat2, lon2, radius_km=6371.0):
    """Distance in km between two points on a sphere."""
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)

    a = (math.sin(delta_phi / 2) ** 2
         + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2) ** 2)

    return 2 * radius_km * math.asin(math.sqrt(a))


# Sanity-check the function on a distance we can look up, before we
# trust it on one we cannot.
london_paris = haversine_km(51.5074, -0.1278, 48.8566, 2.3522)
print("London to Paris: {:.0f} km (published: about 344 km)".format(london_paris))
assert 330 < london_paris < 360, london_paris

The function is right, so now we can use it on the station.

**One leg at a time, not one big total.** It is tempting to add up the
whole distance, divide by the whole time and call it speed. Do not. If a
single reading arrives with a stale timestamp - and free APIs do that -
the aggregate absorbs the error and hands you a confident wrong answer,
with nothing on screen to warn you.

Per-leg speeds cannot hide it. A bad leg stands out next to good ones.

In [ ]:
legs = []

for step in range(len(track) - 1):
    first = track.iloc[step]
    second = track.iloc[step + 1]

    seconds = (second["when"] - first["when"]).total_seconds()
    if seconds <= 0:
        print("skipping leg", step, "- timestamps did not advance")
        continue

    distance = haversine_km(first["latitude"], first["longitude"],
                            second["latitude"], second["longitude"])

    legs.append({
        "seconds": seconds,
        "km": round(distance, 1),
        "km_per_s": round(distance / seconds, 2),
    })

legs = pd.DataFrame(legs)
legs

Look at the spread in that last column before reading on. Are the legs
in agreement, or is one of them arguing with the others?

We take the **median**, which a single bad leg cannot drag around the way
it drags a mean.

In [ ]:
ground_speed = legs["km_per_s"].median()

print("legs measured :", len(legs))
print("slowest leg   : {:.2f} km/s".format(legs["km_per_s"].min()))
print("fastest leg   : {:.2f} km/s".format(legs["km_per_s"].max()))
print("median        : {:.2f} km/s".format(ground_speed))

### From ground speed to orbital speed

Haversine measured the *ground track* - the shadow the station draws on
the Earth's surface, at radius $R = 6371$ km. The station is flying about
420 km higher, on a bigger circle, so in the same time it covers more
distance than its shadow by the ratio of the radii:

$$
v_{\text{orbit}} = v_{\text{ground}} \times \frac{R + h}{R}
$$

In [ ]:
EARTH_RADIUS_KM = 6371.0
ISS_ALTITUDE_KM = 420.0

orbital_speed = ground_speed * (EARTH_RADIUS_KM + ISS_ALTITUDE_KM) / EARTH_RADIUS_KM

print("ground speed  : {:.2f} km/s".format(ground_speed))
print("orbital speed : {:.2f} km/s".format(orbital_speed))
print("              = {:,.0f} km/h".format(orbital_speed * 3600))
print()
print("published     : about 7.66 km/s, 27,600 km/h")
print("our error     : {:+.1%}".format(orbital_speed / 7.66 - 1))

**Expect something in the right neighbourhood, not a precise answer.**
Runs of this notebook have landed anywhere from under 1% to about 8% out.
That is the honest resolution of the method, and the reason is worth
understanding rather than hiding:

- The API reports the time in **whole seconds**. On a 10-second leg, being
  a second out is a 10% error on that leg all by itself.
- Each request takes an unpredictable moment to travel there and back, so
  our gaps are not the neat 10 seconds we asked for.
- We assumed a circular orbit at exactly 420 km. It is neither exactly
  circular nor exactly 420.

Two lessons sit in that, and the second is the one people miss:

1. **Re-run the cells above.** You will get a different answer. A single
   measurement of a noisy thing is not a result, it is a sample.
2. **Precision and accuracy are different.** Python will happily print
   `7.0423 km/s`. Four decimal places of confidence on an input measured
   in whole seconds is a false claim, and it is one of the easiest ways
   to mislead people with entirely correct arithmetic.

If your error is large rather than small, the per-leg table will usually
show you which reading did it. **Knowing why an answer is off is worth
more than getting a tidy one.**

---

## 5. A `200` does not mean the data is right

This is the most important section in the notebook and it takes four
cells.

Same host, different endpoint: who is in space right now.

In [ ]:
ASTROS = "http://api.open-notify.org/astros.json"

astros = fetch(ASTROS, "astros.json")

print("people in space:", astros["number"])
print()
for person in astros["people"]:
    print("  {:<22} {}".format(person["name"], person["craft"]))

Status `200`. Well-formed JSON. Sensible field names. A plausible number
of humans, on plausible spacecraft.

**Now check it against what you know.** Look at those names and that
crew list. Is that who is in orbit this week?

> **Predict first.** Do you believe this list is current? How would you find out, without trusting me or the API?
>
> Put your answer in the chat before we run it.

In [ ]:
# Same host, two endpoints. Ask each one twice, a few seconds apart.
first_position = fetch(ISS_NOW, "iss-now.json")
first_crew = fetch(ASTROS, "astros.json")

time.sleep(5)

second_position = fetch(ISS_NOW, "iss-now.json")
second_crew = fetch(ASTROS, "astros.json")

print()
print("position changed:", first_position["iss_position"] != second_position["iss_position"])
print("crew changed    :", first_crew["people"] != second_crew["people"])
print()
print("position timestamp moved:",
      second_position["timestamp"] - first_position["timestamp"], "seconds")

One endpoint on this host is genuinely live. The other returns a fixed
answer that has not moved in a long time - compare the crew names above
against any current source and you can date the snapshot yourself.

Both return `200`. Both are valid JSON. Both parse cleanly into
dictionaries. **Every check we have built so far passes, and the data is
still out of date.**

Take this away from today:

> `200 OK` means *"I answered you."*
> It does not mean *"this is true"*, and it does not mean *"this is current."*

Freshness is a separate question and you have to ask it deliberately.
Some APIs help - a `Last-Modified` header, a `date` field, a timestamp in
the payload. Where the API does not tell you, **you** have to find a
second source. An analysis is not made correct by the code running
without errors.

In [ ]:
# Where the API gives you a timestamp, use it. Where it does not, that
# absence is itself information.
print("iss-now provides a timestamp:", "timestamp" in first_position)
print("astros provides a timestamp :", "timestamp" in first_crew)
print()
print("astros fields:", sorted(first_crew))

---

## 6. Asking a narrower question, and getting cut off

Real APIs take **parameters**: the same endpoint, refined. Stack Exchange
lets us ask for questions filtered and sorted, with no account at all.

Never build a query string by gluing text together. Hand `requests` a
dictionary and let it do the encoding, including the awkward characters.

In [4]:
SE_QUESTIONS = "https://api.stackexchange.com/2.3/questions"

criteria = {
    "site": "stackoverflow",
    "tagged": "python",
    "sort": "votes",
    "order": "desc",
    "pagesize": 10,
}

popular = fetch(SE_QUESTIONS, "se-questions.json", params=criteria)

for question in popular["items"][:5]:
    print("{:>6}  {}".format(question["score"], question["title"][:66]))

LIVE: https://api.stackexchange.com/2.3/questions?site=stackoverflow&tagged=python&sort=votes&order=d
 13136  What does the &quot;yield&quot; keyword do in Python?
  8443  What does if __name__ == &quot;__main__&quot;: do?
  8128  Does Python have a ternary conditional operator?
  7535  What are metaclasses in Python?
  7351  How do I check whether a file exists without exceptions?


Notice anything odd in those titles? `&quot;` where a double quote should
be. The API returns HTML-escaped text, because it is meant for display in
a browser. Data cleaning starts the moment the data arrives, not later.

In [ ]:
import html

for question in popular["items"][:5]:
    print("{:>6}  {}".format(question["score"],
                             html.unescape(question["title"])[:66]))

### The quota

You are a guest. Stack Exchange tells you, in every single response, how
much welcome you have left.

In [5]:
print("quota_max      :", popular["quota_max"])
print("quota_remaining:", popular["quota_remaining"])
print("has_more       :", popular["has_more"])

quota_max      : 300
quota_remaining: 299
has_more       : True


**300 requests a day** without a key, counted per IP address. Two things
follow that catch people out:

- A whole classroom on one wifi network shares one allowance. Thirty
  students times ten experiments is the daily budget, gone before lunch.
- A loop with a bug can spend the lot in seconds.

Registering for a free key raises the limit substantially. That is the
honest reason most APIs want you identified: not secrecy, but **capacity
planning and a way to switch off whoever is misbehaving.**

When you do run out you get `429 Too Many Requests`, and the polite
response is to wait - which is what the `backoff` field in these
responses is telling you to do.

> **Predict first.** What is the difference, to the server, between one student running a loop 300 times and 300 students running it once?
>
> Put your answer in the chat before we run it.

---

## 7. Wrappers are convenience, not magic

A **wrapper** is a Python library someone wrote around an API, so you
write Python instead of URLs. `stackapi` is one.

It is worth seeing that the wrapper does nothing you could not do
yourself, because sooner or later you will need an API that has no
wrapper, or you will need to work out why the wrapper is misbehaving.

In [ ]:
# The long way: build the parameters, make the call.
raw = fetch(SE_QUESTIONS, "se-questions.json", params={
    "site": "stackoverflow",
    "tagged": "python",
    "sort": "votes",
    "order": "desc",
    "pagesize": 3,
})

print("the long way:")
for question in raw["items"][:3]:
    print("  {:>6}  {}".format(question["score"],
                               html.unescape(question["title"])[:58]))

In [ ]:
from stackapi import StackAPI

# The short way: the wrapper knows the base URL and the site parameter.
site = StackAPI("stackoverflow")
site.page_size = 3
site.max_pages = 1

wrapped = site.fetch("questions", tagged="python", sort="votes", order="desc")

print("the short way:")
for question in wrapped["items"][:3]:
    print("  {:>6}  {}".format(question["score"],
                               html.unescape(question["title"])[:58]))

In [ ]:
long_way = [q["question_id"] for q in raw["items"][:3]]
short_way = [q["question_id"] for q in wrapped["items"][:3]]

print("long way :", long_way)
print("short way:", short_way)
assert long_way == short_way, "these should be the same questions"
print("\nSame questions. Same HTTP request underneath.")

The wrapper saved us the base URL, the `site` parameter, and some
paging. It also spent quota, exactly as our own call did - it is making
the same request over the same protocol to the same endpoint.

What wrappers genuinely buy you: paging handled, rate limits respected,
errors turned into Python exceptions, and results in familiar shapes.
What they cost you: another dependency, and a layer between you and the
thing that is actually going wrong.

---

## 8. When the service needs to know who you are

Everything so far was anonymous. Now we query **BigQuery**, Google's
warehouse for large datasets, which will not talk to strangers.

### How we are authenticating, and why

There are two ways to do this, and the difference is worth your
attention because one of them is how credentials end up on GitHub.

| | Service-account key file | **Application Default Credentials** |
|---|---|---|
| What it is | a `.json` file holding a private key | short-lived tokens managed by `gcloud` |
| Lifetime | until someone revokes it | refreshed automatically |
| Where it lives | wherever you downloaded it | `~/.config/gcloud/`, mode 600 |
| Risk | a file you can email, sync or commit | nothing long-lived to leak |

Google's own console, on the page where you would create a key file,
recommends against creating one. We are using **ADC**. It was set up once
with:

```
gcloud auth application-default login
gcloud auth application-default set-quota-project dsai-teaching-sandbox
```

There is no key file in this folder, and nothing here to add to
`.gitignore`, because the credential is not in the project at all.

Plenty of tutorials - and the lab for this session - use the key-file
route. When you meet it, keep the file outside the repository, never
commit it, and rotate it if it ever moves.

### Cost, which is not optional to think about

BigQuery charges by **bytes scanned**, not rows returned. `SELECT *` on a
large table is an expensive way to look at five rows.

This project runs in the **BigQuery sandbox**: no billing account is
attached, so it *cannot* generate a charge. You get 1 TiB of query
processing a month and 10 GiB of storage. Beyond that, queries fail
rather than bill.

That is the structural protection. We add a second one in code anyway,
because on a project that *does* have billing, the habit is what saves
you.

In [ ]:
from google.cloud import bigquery

PROJECT = "dsai-teaching-sandbox"

client = bigquery.Client(project=PROJECT)   # ADC - note: no key path
print("authenticated against project:", client.project)

### Ask the price before you order

A **dry run** costs nothing and tells you what the real query would
scan. Do this whenever you are about to run something new against a
table you do not know.

In [ ]:
SQL = """
SELECT corpus, SUM(word_count) AS words
FROM `bigquery-public-data.samples.shakespeare`
GROUP BY corpus
ORDER BY words DESC
LIMIT 5
"""

dry = client.query(SQL, job_config=bigquery.QueryJobConfig(
    dry_run=True, use_query_cache=False))

print("would scan {:,} bytes ({:.2f} MB)".format(
    dry.total_bytes_processed, dry.total_bytes_processed / 1024 ** 2))
print("nothing has been scanned yet")

Those backticks around `` `bigquery-public-data.samples.shakespeare` ``
are **not optional**. The project name contains hyphens, and without the
backticks the SQL parser reads them as minus signs. This is the single
most common BigQuery error you will hit.

In [ ]:
job_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=100 * 1024 ** 2,   # hard ceiling: 100 MB
)

shakespeare = client.query(SQL, job_config=job_config).to_dataframe()
shakespeare

A DataFrame, from a warehouse, in three lines. The shape of the work is
identical to section 4 - authenticate, ask, receive, convert.

### Watch the ceiling do its job

`maximum_bytes_billed` is not advice. Set it below what the query needs
and the job is **rejected**, unrun and unbilled.

> **Predict first.** We cap the query below what the dry run said it needs. Does it run and get truncated, or refuse to run at all?
>
> Put your answer in the chat before we run it.

In [ ]:
too_tight = bigquery.QueryJobConfig(maximum_bytes_billed=10 * 1024 ** 2 - 1)

client.query(SQL, job_config=too_tight).result()

`bytesBilledLimitExceeded`. Nothing ran, nothing was billed.

Two details in that error worth keeping:

- The exception class is `InternalServerError`, a `5xx`, which is a
  strange choice for something that is squarely a client-side limit.
  Catch `google.api_core.exceptions.GoogleAPICallError` and you cover it.
- It says a minimum of `10485760` is required. **BigQuery bills a
  minimum of 10 MB per table scanned**, so a cap below that can never be
  satisfied.

In [ ]:
from google.api_core.exceptions import GoogleAPICallError

# use_query_cache=False matters here. We ran this exact SQL a moment ago,
# so BigQuery would serve the cached result, report 0 bytes processed and
# 0 billed, and the point below would be invisible. Caching is free and
# usually what you want - just not when you are measuring.
safe = bigquery.QueryJobConfig(
    maximum_bytes_billed=10 * 1024 ** 2,
    use_query_cache=False,
)

try:
    job = client.query(SQL, job_config=safe)
    rows = job.result()
    print("ran under a 10 MB cap")
    print("  processed: {:,} bytes".format(job.total_bytes_processed))
    print("  billed   : {:,} bytes".format(job.total_bytes_billed))
    print("  rows     :", rows.total_rows)
except GoogleAPICallError as problem:
    print("refused:", type(problem).__name__)

Look at those two numbers. **Processed is smaller than billed**, because
of the 10 MB floor. On a free sandbox that costs nothing. On a real
project, a script that fires thousands of tiny queries is billed for
10 MB every time, and the invoice is a surprise.

---

## 9. An API that answers in prose

Everything so far returned structured data. A generative model returns
language, but it is reached the same way: an endpoint, a key, a request,
a response.

### The key

You need a free key from [aistudio.google.com](https://aistudio.google.com/app/apikey).
**Do not paste it into this notebook.** Anything typed into a cell gets
saved into the file, and this folder is synced.

Put it somewhere outside the project instead:

```
mkdir -p ~/.config/gcp
printf '%s' 'YOUR_KEY_HERE' > ~/.config/gcp/gemini_key.txt
chmod 600 ~/.config/gcp/gemini_key.txt
```

The cell below reads from there, or from a `GEMINI_API_KEY` environment
variable. If it finds neither it says so and the rest of the section is
skipped - the notebook still runs end to end.

In [ ]:
import os

key_file = Path.home() / ".config" / "gcp" / "gemini_key.txt"

GEMINI_KEY = os.environ.get("GEMINI_API_KEY")
if GEMINI_KEY is None and key_file.exists():
    GEMINI_KEY = key_file.read_text().strip()

if GEMINI_KEY is None:
    print("No key found. Section 9 will be skipped.")
    print("Looked for: $GEMINI_API_KEY and", key_file)
else:
    print("Key loaded from outside the project folder.")
    print("Length:", len(GEMINI_KEY), "characters (never print the key itself)")

In [ ]:
from google import genai

if GEMINI_KEY:
    gemini = genai.Client(api_key=GEMINI_KEY)

    available = []
    for model in gemini.models.list():
        if "generateContent" in (model.supported_actions or []):
            available.append(model.name)

    print("models this key can list:", len(available))
    for name in sorted(available)[:8]:
        print("  ", name)
else:
    print("skipped - no key")

### Listed is not the same as callable

That list is what the service will *tell* you about. It is not a promise
that every entry will answer you. Models get retired, and a retired one
can keep appearing in the catalogue while returning `404` to new users.

So we do not hardcode one name and hope. We try candidates in order and
take the first that actually replies - and we treat the two failure modes
differently, because they mean different things:

- **`404`** - this model is gone. Move on to the next one.
- **`503`** - this model exists but is busy right now. Trying again later
  would work; that is a `5xx`, and section 2 told us `5xx` is not your
  fault.

In [ ]:
from google.genai import errors

CANDIDATES = [
    "gemini-3.6-flash",
    "gemini-3.5-flash",
    "gemini-flash-latest",
]

PROMPT = ("In two sentences, explain what an API is to someone who has "
          "never written code.")

reply = None
if GEMINI_KEY:
    for name in CANDIDATES:
        try:
            reply = gemini.models.generate_content(model=name, contents=PROMPT)
            print("answered by:", name)
            break
        except errors.ClientError as problem:
            print("  {} - gone or refused ({})".format(name, problem.code))
        except errors.ServerError as problem:
            print("  {} - service busy ({}), try later".format(name, problem.code))

if reply is not None:
    print()
    print(reply.text)
elif GEMINI_KEY:
    print("\nNo candidate model answered. Check the list above for a "
          "current name.")
else:
    print("skipped - no key")

Same four steps as every other section: point at a service, prove who you
are, send a request, read the response. The body happens to be prose
rather than a table.

Three things to carry out of here:

- **The model is not a source.** It returns fluent text whether or not it
  knows the answer. Section 5 applies with more force here, not less:
  a confident reply is not a checked one.
- **Whatever you send, you have sent.** Free tiers may retain prompts.
  Client data, personal data and health data do not belong in one.
- **Model names expire faster than anything else in this notebook.** The
  loop above exists because a name that worked recently now returns
  `404`. When you copy an example from a blog post, the model name is the
  first thing to check, not the last.

---

## What to take away

1. **A request is a URL.** Scheme, host, path, parameters. Everything
   else is detail.
2. **Check the status before you trust the body.** A 404 does not raise.
3. **`.content` / `.text` / `.json()` are bytes, string and parsed
   object.** Pick deliberately.
4. **`200` means answered, not correct, and not current.** Section 5 is
   the one that will save you professionally.
5. **Quotas are real.** You are a guest on someone's server.
6. **Wrappers are convenience.** Underneath it is the same HTTP call.
7. **Prefer credentials that expire.** ADC over a key file, and never a
   key in a synced folder or a repository.
8. **Ask the price first.** Dry run, then a hard ceiling.

## Try these

1. `http://api.open-notify.org/iss-now.json` returns one position.
   Sample it every 10 seconds for 5 minutes, and plot the ground track
   with `longitude` on the x-axis. Where does the line jump, and why is
   that a plotting artefact rather than a physical one?
2. Stack Exchange has a `/tags` endpoint. Find the 10 most-used tags on
   `datascience.stackexchange.com` and compare them with
   `stackoverflow.com`. Check your quota before and after.
3. Rewrite `get_json()` so it retries twice, waiting a second between
   attempts, but **only** for `5xx` codes. Why would retrying a `4xx` be
   the wrong thing to do?
4. In BigQuery, dry-run a `SELECT *` against
   `` `bigquery-public-data.samples.natality` `` and compare the bytes
   with a query that names three columns. Do not run either for real
   until you have looked at the numbers.

---

*Data Science & AI - Module 3 Part 2. Built by
`tools/build_16_apis.py`. Cached API snapshots live in
`data/api-cache/`.*